a_ is alpha 
m is fuzzifier 
SNI is as given in the text

to do -
1- make a distance function to calculate IFS distance
2- make SNI function which would take in different values of i,j and Neighbourhood matrix N and returns the output

1- N is the neighbourhood matrix(n,n)
2- A is the matrix of all the data point(n,d,3)
3- V is the cluster center matrix(c,d,3)
4- U is the membership Matrix(n,c)
5- Y is the true intensity matrix(n,d,3)

shits got kinda complicated now 
now first i have to compte b diff 
then i have to make another Y matrix and update it continously 
then i will have to make another function which would convert the y matrix accordingly and update it accordingly with regard and respect to the b diff 
wow so here is the pipe line 
make a new Bj diff 
then update the Y matrix such that 
uy(yi) = ux(xi) - bdiff
then using uy(yi) find the values of non membership and hesitation in y
and hence then go and run the update algorithm on U
ik whatever i said for correct formulate that into an algorithm , which i will paste in the markdown 

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def Calculate_lambda(A,U,V,m):
    n = np.size(A,axis=0)
    c = np.size(V,axis=0)
    d = np.size(V,axis=1)

    um = U**m

    term1 = (um@V[:,:,0])/(um.sum(axis=1)[:,None] + 1e-8)
    term1 = A[:,:,0] - term1

    term1 = term1.sum(axis=0)   #check if axis=0
    term2 = um.sum(axis=1)
    term3 = (1/(term2 + 1e-8)).sum()

    return 2*term1/term3

In [ ]:
def Update_B(A,V,U,m):
    um = U**m
    term1 = (um@V[:,:,0])/(um.sum(axis=1)[:,None] + 1e-8)
    term1 = A[:,:,0] - term1
    term2 = Calculate_lambda(A,U,V,m)/(2*(um.sum(axis=1)[:,None] + 1e-8))
    return term1-term2
    

In [ ]:
def Update_Y(A,B,g,beta=None,p=None):
    n = np.size(A,axis=0)
    d = np.size(A,axis=1)
    Y = np.zeros((n,d,3))
    Y[:,:,0] = A[:,:,0] - B
    mu = Y[:,:,0]
    if g == "yager":
        Y[:,:,1] = (1 - mu**p)**(1/p)
    elif g=="sugeno":
        Y[:,:,1] = (1 - mu)/(1 + beta*mu)

    Y[:,:,2] = 1-Y[:,:,0]-Y[:,:,1]

    return Y
    

In [ ]:
def IFS_Distance_num_and_den(Y,V,i,j):
    c = np.size(V,axis=0)
    temp = 0.5 * np.sum((Y[j] - V[i])**2)

    val = 0
    for k in range(c):
        temp = 0.5 * np.sum((Y[j] - V[k])**2)
        val = val + temp

    return temp
    

In [ ]:
def Calculate_SNI(U,A,i,j,N,Y,V):
    list1 = []
    n = np.size(U,axis=0)
    for k in range(n):
        if(N[k][i] ==1):
            list1.append(k)
    val = 0
    for t in list1:
        val = val + U[i][t]

    val2 = 0
    for u in list1:
        temp2 = IFS_Distance_num_and_den(Y,V,i,u)
        val2 = val2 + (U[i][u]*temp2)/(val + 1e-8)
    
    return val2

In [ ]:
# def Calculate_SNI(U,A,i,j,N,Y,V):
#     list1 = []
#     n = np.size(U, axis=1)

#     for k in range(n):
#         if N[j][k] == 1:
#             list1.append(k)

#     val = 0
#     for t in list1:
#         val += U[i][t]

#     val2 = 0
#     for u in list1:
#         temp2 = IFS_Distance_num_and_den(Y, V, i, u)
#         val2 += U[i][u] * temp2

#     return val2 / (val + 1e-8)

In [ ]:
def Update_U(A,V,B,a_,m,N,U,Y):
    U_old = U
    n = np.size(A,axis=0)
    c = np.size(V,axis=0)
    d = np.size(V,axis=1)
    U = np.zeros((n,c))
    for j in range(n):
        for i in range(c):
            temp = IFS_Distance_num_and_den(Y,V,i,j)
            SNI = Calculate_SNI(U_old,A,i,j,N,Y,V)
            den = 0
            for t in range(c):
                temp = IFS_Distance_num_and_den(Y,V,t,j) + a_*Calculate_SNI(U_old,A,t,j,N,Y,V)
                den = den + temp**(-1/(m-1))
            U[i][j] = ((temp + a_*SNI)**(-1/(m-1)))/den
            
            

In [ ]:
def Neighbourhood_Term_inside_membership(U,A,i,j,N,a_):
    list1 = []
    for k_ in range(n):
        if N[k_][j] = 1:
            list1.append(k_)
    den = 0
    for k in list1:
        den = den + U[i][k]

    second_term = 0
    for k in list1:
        second_term = second_term + (U[i][k]*A[k,:,0])/den

    return A[j,:,0] + a_*second_term
        
        
    

In [ ]:
def Neighbourhood_Term_inside_non_membership(U,A,i,j,N,a_):
    list1 = []
    for k_ in range(n):
        if N[k_][j] = 1:
            list1.append(k_)
    den = 0
    for k in list1:
        den = den + U[i][k]

    second_term = 0
    for k in list1:
        second_term = second_term + (U[i][k]*A[k,:,1])/den

    return A[j,:,1] + a_*second_term

In [ ]:
def Neighbourhood_Term_inside_hesitation(U,A,i,j,N,a_):
    list1 = []
    for k_ in range(n):
        if N[k_][j] = 1:
            list1.append(k_)
    den = 0
    for k in list1:
        den = den + U[i][k]

    second_term = 0
    for k in list1:
        second_term = second_term + (U[i][k]*A[k,:,2])/den

    return A[j,:,2] + a_*second_term

In [1]:
def Update_V_membership(U,A,a_,m,N):
    d = np.size(A,axis=1)
    c = np.size(U,axis=1)
    V_membership = np.zeros((c,d))
    um = U**m

    for i in range(c):
        num = 0
        for j in range(n):
            temp = (um[i][j])*(Neighbourhood_Term_inside_membership(U,A,i,j,N,a_))
            num = num + temp
        den = (1+a_)*um[i].sum()
        V_membership[i] = num/den

    return V_membership
    

    
    
    

In [ ]:
def Update_V_hesitation(U,A,a_,m,N):
    d = np.size(A,axis=1)
    c = np.size(U,axis=1)
    V_membership = np.zeros((c,d))
    um = U**m

    for i in range(c):
        num = 0
        for j in range(n):
            temp = (um[i][j])*(Neighbourhood_Term_inside_hesitation(U,A,i,j,N,a_))
            num = num + temp
        den = (1+a_)*um[i].sum()
        V_membership[i] = num/den

    return V_membership

In [ ]:
def Update_V_non_membership(U,A,a_,m,N):
    d = np.size(A,axis=1)
    c = np.size(U,axis=1)
    V_membership = np.zeros((c,d))
    um = U**m

    for i in range(c):
        num = 0
        for j in range(n):
            temp = (um[i][j])*(Neighbourhood_Term_inside_non_membership(U,A,i,j,N,a_))
            num = num + temp
        den = (1+a_)*um[i].sum()
        V_membership[i] = num/den

    return V_membership

In [3]:
def Update_V(U,A,a_,m,N):
    d = np.size(A,axis=1)
    c = np.size(U,axis=1)
    V = np.zeros((c,d,3))
    V[:,:,0] = Update_V_membership(U,A,a_,m,N)
    V[:,:,1] = Update_V_non_membership(U,A,a_,m,N)
    V[:,:,2] = Update_V_hesitation(U,A,a_,m,N)

    return V
    
    

In [ ]:
def Calculate_D_ifs(muA,nuA,piA,muV,nuV,piV,i,j):

    term1 = np.linalg.norm(muA[j] - muV[i])**2
    term2 = np.linalg.norm(nuA[j] - nuV[i])**2
    term3 = np.linalg.norm(piA[j] - piV[i])**2

    return term1 + term2 + term3

In [ ]:
U_old = U.copy()
V_old = V.copy()
B_old = B.copy()
Y_old = Y.copy()



itn = 0

while(itn < 100):
    Y_new = Update_Y(A,B_old,g,beta=None,p=None)

    U_new = Update_U(A,V_old,B_old,a_,m,N,U_old,Y_new)

    V_new = Update_V(U_new,A,a_,m,N)

    B_new = Update_B(A,V_new,U_new,m)

    if Diff(U_old,U_new,V_old,V_new,B_old,B_new) < e:
        break
   
    U_old = U_new.copy()
    B_old = B_new.copy()
    V_old = V_new.copy()
    Y_old = Y_new.copy()

    itn = itn + 1